# 03 — Trajectory exploration

Interactive inspection of diffusion pseudotime, program dynamics, and transitional-cell identification. Use after `run_pipeline.py --step programs` completes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from src.utils import load_config, resolve_path

cfg = load_config()
alv = ad.read_h5ad(resolve_path(cfg['paths']['processed_data']) / 'alveolar_programs.h5ad')
alv

In [ ]:
sc.pl.umap(alv, color=[cfg['conditions']['condition_column'], cfg['cell_types']['annotation_column'], 'dpt_pseudotime'], ncols=3)

In [ ]:
program_cols = [c for c in alv.obs.columns if c.endswith('_score')]
sc.pl.umap(alv, color=program_cols, ncols=3, cmap='viridis')

In [ ]:
# Program trends along pseudotime
import numpy as np
import pandas as pd

bins = np.linspace(0, 1, 21)
alv.obs['pt_bin'] = pd.cut(alv.obs['dpt_pseudotime'], bins=bins, include_lowest=True)
trends = alv.obs.groupby('pt_bin', observed=True)[program_cols].mean()
trends.plot(figsize=(10,5))
plt.xlabel('pseudotime bin'); plt.ylabel('mean program score'); plt.tight_layout()

In [ ]:
# Identify transitional population: high AT2_to_AT1_diff, mid pseudotime, predominantly COVID
trans_mask = (alv.obs['AT2_to_AT1_differentiation_score'] > alv.obs['AT2_to_AT1_differentiation_score'].quantile(0.75)) & \
             (alv.obs['dpt_pseudotime'].between(0.3, 0.7))
trans = alv[trans_mask]
print(f'Transitional: {trans.n_obs} cells')
print(trans.obs[cfg['conditions']['condition_column']].value_counts(normalize=True))